# Classificação de vinhos com Machine Learning

Projeto acadêmico desenvolvido em Python para explorar uma base de dados de características físico-químicas de vinhos e treinar um modelo `ExtraTreesClassifier` para classificar o estilo do vinho (tinto ou branco).

**Etapas:** exploração dos dados com Pandas, visualização, preparação das variáveis, divisão entre treino e teste e avaliação do modelo.

## 1. Bibliotecas

O projeto utiliza Pandas e Matplotlib para análise dos dados e Scikit-learn para o modelo de Machine Learning.

In [ ]:
import io
import zipfile
import urllib.request

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 2. Carregamento da base de dados

A base original do UCI Machine Learning Repository disponibiliza dois arquivos CSV: um para vinhos tintos e outro para vinhos brancos. Os arquivos usam ponto e vírgula como separador. O código abaixo baixa os dados originais e combina os dois conjuntos, adicionando a coluna `style`.

> A planilha Excel recebida para esta atividade passou por conversões automáticas de formato (por exemplo, números decimais transformados em datas ou números sem casas decimais). Para evitar treinar o modelo com valores incorretos, este notebook utiliza os CSVs originais da fonte UCI.

In [ ]:
url = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"

with urllib.request.urlopen(url) as response:
    arquivo_zip = zipfile.ZipFile(io.BytesIO(response.read()))

vinhos_tintos = pd.read_csv(arquivo_zip.open("winequality-red.csv"), sep=";")
vinhos_brancos = pd.read_csv(arquivo_zip.open("winequality-white.csv"), sep=";")

vinhos_tintos["style"] = "red"
vinhos_brancos["style"] = "white"

vinhos = pd.concat([vinhos_tintos, vinhos_brancos], ignore_index=True)
vinhos.head()

## 3. Exploração dos dados

Vamos observar o tamanho da base, os tipos de dados, estatísticas descritivas e a distribuição dos estilos.

In [ ]:
print(f"Linhas: {vinhos.shape[0]} | Colunas: {vinhos.shape[1]}")
display(vinhos.head(10))
display(vinhos.describe())
print("\nTipos de dados:")
print(vinhos.dtypes)
print("\nQuantidade por estilo:")
print(vinhos["style"].value_counts())

In [ ]:
vinhos["quality"].hist(bins=7)
plt.title("Distribuição das notas de qualidade dos vinhos")
plt.xlabel("Qualidade")
plt.ylabel("Quantidade")
plt.show()

## 4. Preparação para o Machine Learning

A variável que queremos prever é `style`. As demais colunas serão utilizadas como características de entrada (`X`). A variável alvo (`y`) será codificada numericamente pelo próprio Scikit-learn usando os rótulos textuais durante o treinamento.

In [ ]:
X = vinhos.drop(columns=["style"])
y = vinhos["style"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Dados de treino:", X_treino.shape)
print("Dados de teste:", X_teste.shape)

## 5. Treinamento e avaliação do modelo

O `ExtraTreesClassifier` combina várias árvores de decisão. Avaliamos o resultado com acurácia, relatório de classificação e matriz de confusão.

In [ ]:
modelo = ExtraTreesClassifier(random_state=42)
modelo.fit(X_treino, y_treino)

previsoes = modelo.predict(X_teste)
acuracia = accuracy_score(y_teste, previsoes)

print(f"Acurácia do modelo: {acuracia:.3f}")
print("\nRelatório de classificação:")
print(classification_report(y_teste, previsoes))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_teste,
    previsoes,
    display_labels=["Tinto", "Branco"],
    cmap="Blues"
)
plt.title("Matriz de confusão")
plt.show()

## 6. Testando uma previsão

A célula abaixo usa uma amostra real da base de teste para demonstrar como o modelo faz uma classificação. Ela não substitui uma validação com novos dados.

In [ ]:
amostra = X_teste.iloc[[0]]
previsao = modelo.predict(amostra)[0]

print("Estilo previsto:", "Vinho tinto" if previsao == "red" else "Vinho branco")
print("Estilo real:", "Vinho tinto" if y_teste.iloc[0] == "red" else "Vinho branco")

## 7. Conclusão

Neste exercício, foi possível praticar análise exploratória com Pandas e construir um modelo de classificação com Scikit-learn. A acurácia e as métricas detalhadas são calculadas durante a execução do notebook.

**Observação:** o desempenho depende da divisão dos dados e das características utilizadas. A classificação do estilo não representa uma avaliação de qualidade ou preferência do consumidor.